In [0]:
import hashlib
import json
from datetime import datetime, timezone

import requests
from pyspark.sql import types as T


# Unity Catalog configuration.
CATALOG = "bucharest_mobility"
BRONZE = "bronze"

TARGET_TABLE = f"{CATALOG}.{BRONZE}.weather_snapshots"

# A single representative point is used for Bucharest.
# City-level weather is sufficient for the hourly transport analysis
# performed by the downstream Gold layer.
LATITUDE = 44.4268
LONGITUDE = 26.1025

API_URL = "https://api.open-meteo.com/v1/forecast"

In [0]:
# Retrieve hourly weather variables relevant to public transport conditions.
# UTC keeps weather timestamps aligned with the normalized GTFS-Realtime data.
#
# Seven historical days provide immediate weather coverage for the collected
# transport observations without requiring a continuously running collector.
params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "hourly": [
        "temperature_2m",
        "precipitation",
        "rain",
        "snowfall",
        "weather_code",
        "wind_speed_10m",
    ],
    "past_days": 7,
    "forecast_days": 1,
    "timezone": "UTC",
}

In [0]:
# One Bronze row represents one complete Open-Meteo API response.
# The raw JSON payload is preserved so parsing and typing remain
# responsibilities of the Silver layer.
bronze_schema = T.StructType([
    T.StructField("location", T.StringType(), False),
    T.StructField("latitude", T.DoubleType(), False),
    T.StructField("longitude", T.DoubleType(), False),
    T.StructField("ingested_at", T.TimestampType(), False),
    T.StructField("payload_hash", T.StringType(), False),
    T.StructField("payload", T.StringType(), False),
])

In [0]:
# Create the Delta table once. Weather snapshots are appended afterward
# so Bronze retains the history returned by the source.
if not spark.catalog.tableExists(TARGET_TABLE):
    empty_df = spark.createDataFrame([], bronze_schema)

    (
        empty_df.write
        .format("delta")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"CREATED {TARGET_TABLE}")
else:
    print(f"EXISTS {TARGET_TABLE}")

In [0]:
# Fetch the weather response and serialize it deterministically.
response = requests.get(
    API_URL,
    params=params,
    timeout=30,
)

response.raise_for_status()

weather_data = response.json()

payload = json.dumps(
    weather_data,
    sort_keys=True,
    separators=(",", ":"),
)

# Content hashing provides a deterministic identifier used to prevent
# identical API responses from being stored more than once.
payload_hash = hashlib.sha256(
    payload.encode("utf-8")
).hexdigest()

print(f"HTTP status: {response.status_code}")
print(f"Payload size: {len(payload):,} characters")
print(f"Hash: {payload_hash}")

In [0]:
# Lightweight source validation before persisting the snapshot.
print("Timezone:", weather_data.get("timezone"))

print(
    "Hourly observations:",
    len(weather_data["hourly"]["time"]),
)

print(
    "First timestamp:",
    weather_data["hourly"]["time"][0],
)

print(
    "Last timestamp:",
    weather_data["hourly"]["time"][-1],
)

In [0]:
# Skip an identical response if its content hash is already present.
already_exists = (
    spark.table(TARGET_TABLE)
    .filter(f"payload_hash = '{payload_hash}'")
    .limit(1)
    .count()
    > 0
)

if already_exists:
    print("SKIP weather snapshot: payload already exists")

else:
    row = [{
        "location": "Bucharest",
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "ingested_at": datetime.now(timezone.utc),
        "payload_hash": payload_hash,
        "payload": payload,
    }]

    df = spark.createDataFrame(
        row,
        schema=bronze_schema,
    )

    # Bronze is append-only: each new source response becomes
    # an immutable historical snapshot.
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )

    print("LOADED weather snapshot")